# FlashML end-to-end Colab quickstart

This notebook trains a tiny text classifier, exports it to ONNX, writes the FlashML bundle files, uploads the bundle, and calls the hosted inference endpoint.

The model is intentionally small so the focus stays on the deployment flow rather than ML training.

## 1. Install dependencies

Colab usually includes PyTorch already. The extra packages below cover tokenization, ONNX export validation, local ONNXRuntime smoke tests, and HTTP upload calls.

In [ ]:
!pip -q install transformers onnx onnxruntime requests

## 2. Train a tiny text classifier

This model uses a Hugging Face tokenizer for text preprocessing and a small hashed bag-of-words classifier for the actual model. The ONNX file stays tiny on purpose, and FlashML will use the same tokenizer settings from `preprocess.json` at inference time.

In [ ]:
from pathlib import Path
import json
import tarfile
import shutil

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

torch.manual_seed(7)

TOKENIZER_NAME = "distilbert-base-uncased"
MAX_LENGTH = 32
NUM_BUCKETS = 128
BUNDLE_DIR = Path("flashml_sentiment_bundle")
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

training_rows = [
    ("I loved the clean dashboard", 1),
    ("The upload flow was fast", 1),
    ("This API saved me time", 1),
    ("The docs were helpful", 1),
    ("The model worked on the first try", 1),
    ("I am happy with the result", 1),
    ("The demo felt reliable", 1),
    ("Inference was quick and simple", 1),
    ("The setup was frustrating", 0),
    ("The upload failed again", 0),
    ("The response was confusing", 0),
    ("I disliked the broken workflow", 0),
    ("The model gave a bad result", 0),
    ("The request kept timing out", 0),
    ("The error message was not useful", 0),
    ("This was slow and unreliable", 0),
]

texts = [row[0] for row in training_rows]
labels = torch.tensor([row[1] for row in training_rows], dtype=torch.long)
encoded = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"], labels)
loader = DataLoader(dataset, batch_size=4, shuffle=True)


class TinyTextClassifier(nn.Module):
    def __init__(self, num_buckets: int = NUM_BUCKETS, num_labels: int = 2):
        super().__init__()
        self.num_buckets = num_buckets
        self.classifier = nn.Linear(num_buckets, num_labels)

    def forward(self, input_ids, attention_mask):
        buckets = torch.remainder(input_ids, self.num_buckets)
        token_features = torch.nn.functional.one_hot(buckets, num_classes=self.num_buckets).float()
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (token_features * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.classifier(pooled)


model = TinyTextClassifier()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-2)
loss_fn = nn.CrossEntropyLoss()

model.train()
for epoch in range(60):
    total_loss = 0.0
    for input_ids, attention_mask, batch_labels in loader:
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"epoch={epoch + 1} loss={total_loss / len(loader):.4f}")

## 3. Export to ONNX

FlashML text inference expects the ONNX input names to match tokenizer outputs. This model accepts `input_ids` and `attention_mask`, which are produced by the tokenizer configured in `preprocess.json`.

In [ ]:
model.eval()
onnx_path = BUNDLE_DIR / "model.onnx"

sample = tokenizer(
    "The upload flow was fast and reliable",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

export_args = dict(
    model=model,
    args=(sample["input_ids"], sample["attention_mask"]),
    f=onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch"},
        "attention_mask": {0: "batch"},
        "logits": {0: "batch"},
    },
    opset_version=13,
    do_constant_folding=False,
    external_data=False,
)

try:
    torch.onnx.export(**export_args, dynamo=False)
except TypeError:
    export_args.pop("external_data", None)
    try:
        torch.onnx.export(**export_args, dynamo=False)
    except TypeError:
        torch.onnx.export(**export_args)

external_data_path = onnx_path.with_suffix(onnx_path.suffix + ".data")
if external_data_path.exists():
    raise RuntimeError(
        f"Unexpected external ONNX data file created: {external_data_path}. "
        "FlashML text classification examples should export a single-file ONNX model."
    )

print(f"Wrote {onnx_path}")

## 4. Write FlashML bundle files

A text classification bundle needs one `.onnx` file and one `preprocess.json`. `labels.txt` is optional but recommended so responses return readable labels instead of numeric class IDs. The tokenizer files are saved too, which makes validation independent of downloading tokenizer assets later.

In [ ]:
preprocess = {
    "tokenizer": TOKENIZER_NAME,
    "max_length": MAX_LENGTH,
    "padding": "max_length",
    "truncation": True,
}

(BUNDLE_DIR / "preprocess.json").write_text(json.dumps(preprocess, indent=2))
(BUNDLE_DIR / "labels.txt").write_text("negative\npositive\n")
tokenizer.save_pretrained(BUNDLE_DIR)

print("Bundle directory:")
for path in sorted(BUNDLE_DIR.iterdir()):
    print("-", path.name)

## 5. Smoke test the ONNX model locally

This is optional, but it catches mismatched input names, broken exports, and output shapes that would fail FlashML's text-classification post-processing path.

In [ ]:
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("ONNX inputs:", [(inp.name, inp.type, inp.shape) for inp in session.get_inputs()])
print("ONNX outputs:", [(out.name, out.type, out.shape) for out in session.get_outputs()])

def predict_local(text: str):
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    raw_output = session.run(None, {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64),
    })[0]
    print("Raw output shape:", raw_output.shape)

    # Mirrors FlashML's text-classification post-processing.
    logits = raw_output[0]
    if logits.ndim != 1:
        raise ValueError(f"Expected one logit vector, got shape {raw_output.shape}")
    probs = np.exp(logits) / np.exp(logits).sum()
    label_names = ["negative", "positive"]
    return {label_names[i]: float(probs[i]) for i in range(len(label_names))}

predict_local("The docs were clear and the upload worked")

## 6. Package the model bundle

FlashML expects one gzipped tar archive. The archive may contain a top-level folder, but this example keeps files at the archive root.

In [ ]:
bundle_path = Path("flashml-sentiment-classifier.tar.gz")

with tarfile.open(bundle_path, "w:gz") as tar:
    for path in sorted(BUNDLE_DIR.iterdir()):
        tar.add(path, arcname=path.name)

print(f"Created {bundle_path} ({bundle_path.stat().st_size / 1024:.1f} KB)")
print("A small bundle is expected here because this demo model uses hashed token features, not a large embedding table.")

## 7. Upload to FlashML

Create an API key in FlashML, then paste it when prompted. The notebook uses `getpass` so the key is not displayed in cell output.

In [ ]:
from getpass import getpass
import requests

BASE_URL = "https://api.flashml.dev"
API_KEY = getpass("FlashML API key: ")
headers = {"Authorization": f"Bearer {API_KEY}"}

reservation = requests.post(
    f"{BASE_URL}/v1/models/uploads",
    headers={**headers, "Content-Type": "application/json"},
    json={
        "filename": bundle_path.name,
        "modality": "text",
        "task": "classification",
        "size_bytes": bundle_path.stat().st_size,
    },
    timeout=30,
)
print(reservation.status_code, reservation.text)
reservation.raise_for_status()
upload = reservation.json()

with bundle_path.open("rb") as file:
    put = requests.put(upload["upload_url"], data=file, timeout=300)
print("PUT", put.status_code)
put.raise_for_status()

completed = requests.post(
    f"{BASE_URL}/v1/models/uploads/{upload['upload_id']}/complete",
    headers=headers,
    timeout=300,
)
print(completed.status_code, completed.text)
completed.raise_for_status()

result = completed.json()
model_id = result["model"]["id"]
print("FlashML model ID:", model_id)

## 8. Run hosted inference

When completion returns a model with `status: validated`, there is no separate deploy step. Use the returned model ID immediately.

In [ ]:
response = requests.post(
    f"{BASE_URL}/v1/models/{model_id}/infer",
    headers={**headers, "Content-Type": "application/json"},
    json={"text": "The FlashML upload flow was simple and reliable."},
    timeout=60,
)
print(response.status_code)
print(response.json())
response.raise_for_status()